# **MÓDULO 35 - Cross Validation**

Nesta tarefa, você trabalhará com uma base de dados que contém informações sobre variáveis ambientais coletadas para a detecção de incêndios. O objetivo é utilizar técnicas de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação na previsão da ocorrência de um incêndio com base nas variáveis fornecidas.


Descrição da Base de Dados
A base de dados contém as seguintes variáveis:

Unnamed:0: Índice (não é uma variável útil para o modelo)

UTC: Tempo em Segundos UTC

Temperature[C]: Temperatura do Ar (em graus Celsius)

Humidity[%]: Umidade do Ar (em porcentagem)

TVOC[ppb]: Total de Compostos Orgânicos Voláteis (medido em partes por bilhão)

eCO2[ppm]: Concentração equivalente de CO2 (medido em partes por milhão)

Raw H2: Hidrogênio molecular bruto, não compensado

Raw Ethanol: Etanol gasoso bruto

Pressure[hPA]: Pressão do Ar (em hectopascais)

PM1.0: Material particulado de tamanho < 1,0 µm

PM2.5: Material particulado de tamanho >1,0 µm e < 2,5 µm

NC0.5: Concentração numérica de material particulado de tamanho < 0,5 µm

NC1.0: Concentração numérica de material particulado de tamanho 0,5 µm < 1,0 µm

NC2.5: Concentração numérica de material particulado de tamanho 1,0 µm < 2,5 µm

CNT: Contador de amostras


E a variável alvo:

Fire Alarm: Indicador binário de incêndio (1 se houver incêndio, 0 caso contrário)

O objetivo desta tarefa é aplicar a técnica de validação cruzada (cross-validation) para avaliar a performance de um modelo de classificação. A validação cruzada ajudará a garantir que o modelo seja avaliado de maneira robusta e generalize bem para dados não vistos.

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

# 1 - Carregue a base de dados, verifique os tipos de dados e também se há presença de dados faltantes ou nulos.

In [2]:
df = pd.read_csv('/content/Cientista de dados M35 - smoke_detection_iot.csv')
df.head()

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


In [9]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

In [13]:
 # Estatísticas básicas

df.describe()

,unnamed:_0,utc,temperature[c],humidity[%],tvoc[ppb],eco2[ppm],raw_h2,raw_ethanol,pressure[hpa],pm1.0,pm2.5,nc0.5,nc1.0,nc2.5,cnt,fire_alarm
count,62630.000000,6.263000e+04,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000,62630.000000
mean,31314.500000,1.654792e+09,15.970424,48.539499,1942.057528,670.021044,12942.453936,19754.257912,938.627649,100.594309,184.467770,491.463608,203.586487,80.049042,10511.386157,0.714626
std,18079.868017,1.100025e+05,14.359576,8.865367,7811.589055,1905.885439,272.464305,609.513156,1.331344,922.524245,1976.305615,4265.661251,2214.738556,1083.383189,7597.870997,0.451596
min,0.000000,1.654712e+09,-22.010000,10.740000,0.000000,400.000000,10668.000000,15317.000000,930.852000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,15657.250000,1.654743e+09,10.994250,47.530000,130.000000,400.000000,12830.000000,19435.000000,938.700000,1.280000,1.340000,8.820000,1.384000,0.033000,3625.250000,0.000000
50%,31314.500000,1.654762e+09,20.130000,50.150000,981.000000,400.000000,12924.000000,19501.000000,938.816000,1.810000,1.880000,12.450000,1.943000,0.044000,9336.000000,1.000000
75%,46971.750000,1.654778e+09,25.409500,53.240000,1189.000000,438.000000,13109.000000,20078.000000,939.418000,2.090000,2.180000,14.420000,2.249000,0.051000,17164.750000,1.000000
max,62629.000000,1.655130e+09,59.930000,75.200000,60000.000000,60000.000000,13803.000000,21410.000000,939.861000,14333.690000,45432.260000,61482.030000,51914.680000,30026.438000,24993.000000,1.000000


In [11]:
# Informações Gerais
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   unnamed:_0      62630 non-null  int64  
 1   utc             62630 non-null  int64  
 2   temperature[c]  62630 non-null  float64
 3   humidity[%]     62630 non-null  float64
 4   tvoc[ppb]       62630 non-null  int64  
 5   eco2[ppm]       62630 non-null  int64  
 6   raw_h2          62630 non-null  int64  
 7   raw_ethanol     62630 non-null  int64  
 8   pressure[hpa]   62630 non-null  float64
 9   pm1.0           62630 non-null  float64
 10  pm2.5           62630 non-null  float64
 11  nc0.5           62630 non-null  float64
 12  nc1.0           62630 non-null  float64
 13  nc2.5           62630 non-null  float64
 14  cnt             62630 non-null  int64  
 15  fire_alarm      62630 non-null  int64  
dtypes: float64(8), int64(8)
memory usage: 7.6 MB


O dataset possui 62.630 registros e 16 colunas, todas sem valores nulos.

Para a coluna Fire Alarm, por conta do espaçamento talvez seja util renomear o nome da coluna utilizando:

df.rename(columns={'Fire Alarm': 'Fire_Alarm'}, inplace=True)

# 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.




# Escolha: Regressão Logística

- Isso porque a variável alvo (fire_alarm) é binária (0 ou 1), ou seja, trata-se de um problema de classificação. A Regressão Logística é simples, eficiente e funciona muito bem como baseline para esse tipo de tarefa.


---


Além disso, como existem várias variáveis numéricas relacionadas a sensores ambientais, o modelo consegue aprender bem padrões lineares entre essas variáveis e a ocorrência de alarme.

# 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará.

In [14]:
# separando variáveis
X = df.drop('fire_alarm', axis=1)
y = df['fire_alarm']

# instanciando o modelo
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

# 4 - Defina o número de Folds e rode o modelo com a validação cruzada.

In [16]:
from sklearn.model_selection import cross_validate

k = 5

scoring = ['accuracy', 'precision', 'recall', 'f1']

results = cross_validate(model, X, y, cv=k, scoring=scoring)

print("=== RESULTADOS POR FOLD ===\n")

for i in range(k):
    print(f"Fold {i+1}")
    print(f"Acurácia: {results['test_accuracy'][i]:.4f}")
    print(f"Precisão: {results['test_precision'][i]:.4f}")
    print(f"Recall:   {results['test_recall'][i]:.4f}")
    print(f"F1-score: {results['test_f1'][i]:.4f}")
    print("-" * 30)

=== RESULTADOS POR FOLD ===

Fold 1
Acurácia: 0.8038
Precisão: 0.7846
Recall:   1.0000
F1-score: 0.8793
------------------------------
Fold 2
Acurácia: 0.9717
Precisão: 0.9619
Recall:   1.0000
F1-score: 0.9806
------------------------------
Fold 3
Acurácia: 0.9284
Precisão: 0.9999
Recall:   0.8999
F1-score: 0.9473
------------------------------
Fold 4
Acurácia: 1.0000
Precisão: 1.0000
Recall:   1.0000
F1-score: 1.0000
------------------------------
Fold 5
Acurácia: 0.8998
Precisão: 0.9837
Recall:   0.8743
F1-score: 0.9258
------------------------------


# 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.

In [17]:
print("=== FOLD 5 ===")
print(f"Acurácia: {results['test_accuracy'][4]:.4f}")
print(f"Precisão: {results['test_precision'][4]:.4f}")
print(f"Recall:   {results['test_recall'][4]:.4f}")
print(f"F1-score: {results['test_f1'][4]:.4f}")

=== FOLD 5 ===
Acurácia: 0.8998
Precisão: 0.9837
Recall:   0.8743
F1-score: 0.9258


# Avaliação dos Resultados de Cada Fold

- **Fold 1** apresentou acurácia de 0,8038, com precisão de 0,7846 e F1 de 0,8793, mostrando desempenho mais fraco em comparação aos demais, apesar de recall perfeito.
- **Fold 2** teve forte melhora, com acurácia de 0,9717 e F1 de 0,9806, indicando ótimo equilíbrio entre as métricas.
- **Fold 3** manteve bom desempenho (acurácia 0,9284), mas com recall menor (0,8999), indicando alguns erros na detecção de positivos.
- **Fold 4** foi o melhor cenário possível, com 1,0000 em todas as métricas, sugerindo separação perfeita nesse subconjunto.
- **Fold 5** apresentou desempenho intermediário, com acurácia de 0,8998 e F1 de 0,9258, mostrando leve queda em relação aos melhores folds.

# Validação final (média dos folds):

- Acurácia média: 0,9207
- Precisão média: 0,9460
- Recall médio: 0,9548
- F1-score médio: 0,9466

O modelo apresenta desempenho geral muito bom, com alta capacidade de classificação e equilíbrio entre precisão e recall. Porém, a variação entre os folds indica certa instabilidade, sugerindo que o modelo pode depender da forma como os dados são divididos.